In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **3-CHANNELS-TEXT-DATA-PREPROCESSING**

In [4]:
import numpy as np
import mne
import os
import matplotlib.pyplot as plt

# Function to preprocess a single .txt file
def preprocess_txt_file(file_path, sfreq):
    data = np.loadtxt(file_path)
    ch_names = ['Channel1', 'Channel2', 'Channel3']
    ch_types = ['eeg'] * 3
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
    raw = mne.io.RawArray(data.T, info)
    print(f"Loaded {file_path}")

    # Apply bandpass filtering
    raw.filter(l_freq=1.0, h_freq=30.0)
    print("Bandpass filtering (1.0 - 30.0 Hz) applied.")

    # Set EEG reference to average
    raw.set_eeg_reference('average')
    print("Re-referencing to average applied.")

    # Define a custom montage for plotting
    custom_montage = mne.channels.make_dig_montage({
        'Channel1': [0.0, -0.1, 0.0],  # Example positions
        'Channel2': [0.1, 0.0, 0.0],
        'Channel3': [0.0, 0.1, 0.0]
    }, coord_frame='head')

    raw.set_montage(custom_montage)
    print("Custom montage applied.")

    # Plot electrode placement in 2D and 3D
    fig1 = raw.plot_sensors(kind='3d', show_names=True, show=False)
    fig2 = raw.plot_sensors(kind='topomap', show_names=True, show=False)
    
    # Save the plots
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    plot_path_3d = os.path.join(output_dir, f"{base_name}_3d.png")
    plot_path_topomap = os.path.join(output_dir, f"{base_name}_topomap.png")
    fig1.savefig(plot_path_3d)
    fig2.savefig(plot_path_topomap)
    plt.close(fig1)
    plt.close(fig2)
    print(f"Saved electrode placement plots for {file_path}")

    # Epoching
    epochs = mne.make_fixed_length_epochs(raw, duration=2.0, preload=True)
    print(f"Epochs created with {len(epochs)} epochs.")

    return epochs

# Directory containing .txt files
input_dir = '/kaggle/input/modma-dataset/EEG_3channels_resting_lanzhou_2015/EEG_3channels_resting_lanzhou_2015/'
output_dir = '/kaggle/working/preprocessed_text_output'
os.makedirs(output_dir, exist_ok=True)

# Assuming a common sampling rate
sfreq = 250

# Process all .txt files in the directory
for file_name in os.listdir(input_dir):
    if file_name.endswith('.txt'):
        file_path = os.path.join(input_dir, file_name)
        try:
            epochs = preprocess_txt_file(file_path, sfreq)
            output_path = os.path.join(output_dir, file_name.replace('.txt', '-epo.fif'))
            epochs.save(output_path, overwrite=True)
            print(f"Saved preprocessed data to {output_path}")
        except Exception as e:
            print(f"Skipping file {file_name} due to preprocessing errors: {e}")


Creating RawArray with float64 data, n_channels=3, n_times=450436
    Range : 0 ... 450435 =      0.000 ...  1801.740 secs
Ready.
Loaded /kaggle/input/modma-dataset/EEG_3channels_resting_lanzhou_2015/EEG_3channels_resting_lanzhou_2015/02030007_still.txt
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 825 samples (3.300 s)

Bandpass filtering (1.0 - 30.0 Hz) applied.
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Re-referencing to average appli

# **128-Channel ERP-EEG Dataset (.raw Files)**

In [ ]:
import mne
import os
import matplotlib.pyplot as plt

def preprocess_raw_file(file_path, output_dir):
    try:
        # Load the raw file
        raw = mne.io.read_raw_egi(file_path, preload=True)
        print(f"Loaded {file_path}")

        # Apply bandpass filtering
        raw.filter(l_freq=1.0, h_freq=30.0)
        print("Bandpass filtering (1.0 - 30.0 Hz) applied.")

        # Set EEG reference to average
        raw.set_eeg_reference('average')
        print("Re-referencing to average applied.")

        # Apply standard montage
        montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
        
        # Check for missing channels and rename if necessary
        missing_chs = [ch for ch in raw.ch_names if ch not in montage.ch_names]
        if missing_chs:
            print(f"Missing channels in montage: {missing_chs}")
            raw.info['bads'] += missing_chs
            raw.drop_channels(missing_chs)
        
        raw.set_montage(montage)
        print("Standard montage 'GSN-HydroCel-128' applied.")
        
        # Plot electrode placement in 2D and 3D
        fig = raw.plot_sensors(kind='3d')
        plt.show()  # Show the plot
        fig2 = raw.plot_sensors(kind='topomap', show_names=False)
        plt.show()  # Show the plot

        # Save the preprocessed file
        output_file_name = os.path.splitext(os.path.basename(file_path))[0] + '-preprocessed.fif'
        output_path = os.path.join(output_dir, output_file_name)
        raw.save(output_path, overwrite=True)
        print(f"Saved preprocessed data to {output_path}")

        return raw

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Directory containing .raw files
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_ERP_lanzhou_2015/EEG_128channels_ERP_lanzhou_2015'
output_dir = '/kaggle/working/preprocessed_raw_output_directory'
os.makedirs(output_dir, exist_ok=True)

# Process all .raw files in the directory
for file_name in os.listdir(input_dir):
    if file_name.endswith('.raw'):
        file_path = os.path.join(input_dir, file_name)
        raw = preprocess_raw_file(file_path, output_dir)
        if raw:
            # Save the preprocessed file
            output_file_name = file_name.replace('.raw', '-preprocessed.fif')
            output_path = os.path.join(output_dir, output_file_name)
            raw.save(output_path, overwrite=True)
            print(f"Saved preprocessed data to {output_path}")
        else:
            print(f"Skipping {file_name} due to errors.")


# **128-Channel ERP-EEG Dataset (.mat Files)**

In [ ]:
import scipy.io
import mne
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import signal

def adaptive_noise_cancellation_lms(raw, reference_idx, mu=0.01, n_iter=1):
    """Apply LMS adaptive noise cancellation."""
    data = raw.get_data()
    ref = data[reference_idx, :]
    
    for _ in range(n_iter):
        for i in range(data.shape[0]):
            if i != reference_idx:
                error = data[i, :] - ref
                weights = mu * error * ref
                data[i, :] = data[i, :] - weights
                
    raw._data = data
    return raw

def preprocess_mat_file(file_path, output_dir):
    try:
        # Load .mat file
        mat = scipy.io.loadmat(file_path)
        print(f"Loaded {file_path}")

        # Inspect the structure
        keys = list(mat.keys())
        print("Keys in .mat file:", keys)

        # Find the correct key for the EEG data
        data_key = None
        for key in keys:
            if 'Impedances_0' in key:
                data_key = key
                break

        if data_key is None:
            raise KeyError("EEG data key not found in the .mat file.")

        # Extract data
        data = mat[data_key]

        # Check data dimensions and transpose if necessary
        if data.shape[0] > data.shape[1]:
            data = data.T

        # Create MNE info structure
        sfreq = 250  # Sampling frequency set to 250 Hz
        ch_names = [f'EEG {i+1:03d}' for i in range(data.shape[0])]
        info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')

        # Create RawArray object
        raw = mne.io.RawArray(data, info)
        print("Raw data created.")

        # Apply bandpass filtering
        raw.filter(l_freq=1.0, h_freq=30.0)
        print("Bandpass filtering (1.0 - 30.0 Hz) applied.")

        # Apply LMS-based blink artifact removal
        raw = adaptive_noise_cancellation_lms(raw, reference_idx=0)  # Adjust reference index as needed
        print("LMS-based artifact removal applied.")

        # Set EEG reference to average
        raw.set_eeg_reference('average')
        print("Re-referencing to average applied.")

        # Apply standard montage
        montage = mne.channels.make_standard_montage('GSN-HydroCel-128')

        # Match and rename channels if needed
        missing_chs = [ch for ch in raw.ch_names if ch not in montage.ch_names]
        if missing_chs:
            print(f"Missing channels in montage: {missing_chs}")
            # Rename channels if needed
            rename_dict = {ch: montage.ch_names[i] for i, ch in enumerate(raw.ch_names) if ch not in montage.ch_names}
            raw.rename_channels(rename_dict)
            # Drop channels still missing after renaming
            missing_chs_after_rename = [ch for ch in raw.ch_names if ch not in montage.ch_names]
            raw.drop_channels(missing_chs_after_rename)
        raw.set_montage(montage)
        print("Standard montage 'GSN-HydroCel-128' applied.")

        # Plot electrode placement in 2D and 3D
        fig1 = raw.plot_sensors(kind='3d', show_names=True, show=False)
        fig2 = raw.plot_sensors(kind='topomap', show_names=False, show=False)
        
        # Save the plots
        plot_path_3d = os.path.join(output_dir, f"{os.path.splitext(os.path.basename(file_path))[0]}_3d.png")
        plot_path_topomap = os.path.join(output_dir, f"{os.path.splitext(os.path.basename(file_path))[0]}_topomap.png")
        fig1.savefig(plot_path_3d)
        fig2.savefig(plot_path_topomap)
        plt.close(fig1)
        plt.close(fig2)

        # Define events manually if needed or use a placeholder
        events = np.array([[i, 0, 1] for i in range(0, raw.n_times, int(sfreq))])
        event_id = {'Stimulus': 1}  # Placeholder event ID

        # Epoching
        epochs = mne.Epochs(raw, events=events, event_id=event_id, tmin=-0.2, tmax=0.8, preload=True)
        print(f"Epochs created with {len(epochs)} epochs.")

        # Save the epochs
        output_file_name = os.path.splitext(os.path.basename(file_path))[0] + '-epo.fif'
        output_path = os.path.join(output_dir, output_file_name)
        epochs.save(output_path, overwrite=True)
        print(f"Saved preprocessed data to {output_path}")

        return epochs
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Directory containing .mat files
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_resting_lanzhou_2015/EEG_128channels_resting_lanzhou_2015'
output_dir = '/kaggle/working/preprocessed_mat_output_directory'
os.makedirs(output_dir, exist_ok=True)

# Process all .mat files in the directory
for file_name in os.listdir(input_dir):
    if file_name.endswith('.mat'):
        file_path = os.path.join(input_dir, file_name)
        epochs = preprocess_mat_file(file_path, output_dir)
        if epochs:
            print(f"Preprocessing complete for {file_name}.")
        else:
            print(f"Skipping {file_name} due to errors.")


In [ ]:
epochs._get_data()

In [ ]:
epochs.plot_drop_log()

In [ ]:
import scipy.io as sio

file_path = '/kaggle/input/modma-dataset/EEG_128channels_resting_lanzhou_2015/EEG_128channels_resting_lanzhou_2015/02010005rest 20150507 0907..mat'  # Update this path
mat_contents = sio.loadmat(file_path)
print(mat_contents.keys())  # Print all keys in the MAT file
